# EDA 009: Game Prior Aggregates (Leakage-Safe)

Goal: build game-history features for each review using only records with `timestamp_created` strictly earlier than the current row.

In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

TRAIN_PATH = Path("../../data/interim/steam_reviews_cleaned_english_train.parquet")
VAL_PATH = Path("../../data/interim/steam_reviews_cleaned_english_val.parquet")
TEST_PATH = Path("../../data/interim/steam_reviews_cleaned_english_test.parquet")
OUT_DIR = Path("../../data/interim/features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLS = [
    "review_id",
    "app_id",
    "timestamp_created",
    "recommended",
    "votes_helpful",
    "review_length_chars",
]

In [2]:
def _load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{split_name} missing required columns: {missing}")
    out = df[REQUIRED_COLS].copy()
    out["split"] = split_name
    return out


train = _load_split(TRAIN_PATH, "train")
val = _load_split(VAL_PATH, "val")
test = _load_split(TEST_PATH, "test")

full = pd.concat([train, val, test], ignore_index=True)
full["timestamp_created"] = full["timestamp_created"].astype("int64")
full["recommended_int"] = full["recommended"].astype("int64")
full["review_length_chars"] = full["review_length_chars"].fillna(0).astype("float64")
full["votes_helpful"] = full["votes_helpful"].fillna(0).astype("float64")

full.shape

(9160492, 8)

In [3]:
def build_game_priors_strict(df: pd.DataFrame) -> pd.DataFrame:
    keys = ["app_id", "timestamp_created"]
    ts_agg = (
        df.groupby(keys, as_index=False)
        .agg(
            ts_count=("review_id", "size"),
            ts_rec_sum=("recommended_int", "sum"),
            ts_help_sum=("votes_helpful", "sum"),
            ts_len_sum=("review_length_chars", "sum"),
        )
        .sort_values(keys)
    )

    g = ts_agg.groupby("app_id", sort=False)
    ts_agg["game_prior_review_count"] = g["ts_count"].cumsum()
    ts_agg["game_prior_rec_sum"] = g["ts_rec_sum"].cumsum()
    ts_agg["game_prior_help_sum"] = g["ts_help_sum"].cumsum()
    ts_agg["game_prior_len_sum"] = g["ts_len_sum"].cumsum()
    ts_agg["game_prior_review_count"] = g["game_prior_review_count"].shift(1).fillna(0)
    ts_agg["game_prior_rec_sum"] = g["game_prior_rec_sum"].shift(1).fillna(0)
    ts_agg["game_prior_help_sum"] = g["game_prior_help_sum"].shift(1).fillna(0)
    ts_agg["game_prior_len_sum"] = g["game_prior_len_sum"].shift(1).fillna(0)
    ts_agg["game_prev_timestamp"] = g["timestamp_created"].shift(1)

    denom = ts_agg["game_prior_review_count"].replace(0, np.nan)
    ts_agg["game_prior_recommend_rate"] = (ts_agg["game_prior_rec_sum"] / denom).fillna(0.0)
    ts_agg["game_prior_mean_votes_helpful"] = (ts_agg["game_prior_help_sum"] / denom).fillna(0.0)
    ts_agg["game_prior_mean_review_len"] = (ts_agg["game_prior_len_sum"] / denom).fillna(0.0)
    ts_agg["game_seconds_since_last_review"] = (
        ts_agg["timestamp_created"] - ts_agg["game_prev_timestamp"]
    ).fillna(-1)

    merge_cols = [
        "app_id",
        "timestamp_created",
        "game_prior_review_count",
        "game_prior_recommend_rate",
        "game_prior_mean_votes_helpful",
        "game_prior_mean_review_len",
        "game_seconds_since_last_review",
    ]
    out = df.merge(ts_agg[merge_cols], on=["app_id", "timestamp_created"], how="left")
    out["game_prior_review_count"] = out["game_prior_review_count"].fillna(0).astype("int64")
    for c in [
        "game_prior_recommend_rate",
        "game_prior_mean_votes_helpful",
        "game_prior_mean_review_len",
        "game_seconds_since_last_review",
    ]:
        out[c] = out[c].fillna(0.0)
    return out


game_feat = build_game_priors_strict(full)
game_feat[["review_id", "split", "game_prior_review_count", "game_prior_recommend_rate"]].head()

,review_id,split,game_prior_review_count,game_prior_recommend_rate
0,85184605,train,146765,0.976493
1,85184171,train,146764,0.976493
2,85184064,train,146763,0.976493
3,85180436,train,146762,0.976493
4,85179753,train,146761,0.976492


In [4]:
# Leakage checks and spot checks
first_rows = (
    game_feat.sort_values(["app_id", "timestamp_created", "review_id"])
    .groupby("app_id", as_index=False)
    .head(1)
)
assert (first_rows["game_prior_review_count"] == 0).all(), "First game row should have zero prior count"

same_ts_counts = (
    game_feat.groupby(["app_id", "timestamp_created"])["game_prior_review_count"].nunique().max()
)
assert same_ts_counts == 1, "Rows with same timestamp must share identical strict-prior features"

sample = game_feat.sample(n=min(10, len(game_feat)), random_state=42)[
    [
        "review_id",
        "app_id",
        "timestamp_created",
        "game_prior_review_count",
        "game_prior_recommend_rate",
        "game_prior_mean_votes_helpful",
    ]
]
sample

,review_id,app_id,timestamp_created,game_prior_review_count,game_prior_recommend_rate,game_prior_mean_votes_helpful
7761957,30224160,418370,1488216657,4595,0.931447,4.523830
6689153,19983304,227300,1451187376,15049,0.976410,2.609409
7534727,72272304,322330,1594179623,55354,0.963345,1.279402
2389987,41472153,552520,1523671182,6865,0.732265,5.576111
4729440,68157709,346110,1588049905,144807,0.715276,3.113876
2066135,77201864,945360,1602110866,120769,0.959518,1.169704
4035568,70212575,271590,1591136228,252199,0.709618,2.701728
8714972,39145619,578080,1515316293,134395,0.647941,2.883552
1302006,17038685,227300,1436850690,12199,0.978933,2.817116
2821539,65447474,203160,1584766383,38626,0.944234,1.064594


In [9]:
game_feat.sort_values(["app_id", "timestamp_created", "review_id"]).head(10)

,review_id,app_id,timestamp_created,recommended,votes_helpful,review_length_chars,split,recommended_int,game_prior_review_count,game_prior_recommend_rate,game_prior_mean_votes_helpful,game_prior_mean_review_len,game_seconds_since_last_review
6437159,1771521,70,1290809452,1,0.0,1138.0,val,1,0,0.0,0.000000,0.000000,-1.0
121242,1865781,70,1290811045,1,10.0,51.0,train,1,1,1.0,0.000000,1138.000000,1593.0
121241,3514704,70,1290885135,1,1.0,24.0,train,1,2,1.0,5.000000,594.500000,74090.0
121240,87268,70,1290915686,1,0.0,108.0,train,1,3,1.0,3.666667,404.333333,30551.0
121239,878042,70,1290919485,1,0.0,552.0,train,1,4,1.0,2.750000,330.250000,3799.0
7811763,119565,70,1290940303,1,1.0,97.0,test,1,5,1.0,2.200000,374.600000,20818.0
6437158,113349,70,1290942922,1,0.0,229.0,val,1,6,1.0,2.000000,328.333333,2619.0
121238,619160,70,1291014464,1,0.0,156.0,train,1,7,1.0,1.714286,314.142857,71542.0
121237,1111054,70,1291021193,1,3.0,60.0,train,1,8,1.0,1.500000,294.375000,6729.0
121236,1062366,70,1291056961,1,1.0,41.0,train,1,9,1.0,1.666667,268.333333,35768.0


In [ ]:
# feature_cols = [
#     "review_id",
#     "game_prior_review_count",
#     "game_prior_recommend_rate",
#     "game_prior_mean_votes_helpful",
#     "game_prior_mean_review_len",
#     "game_seconds_since_last_review",
# ]

# for split_name in ["train", "val", "test"]:
#     split_out = game_feat.loc[game_feat["split"] == split_name, feature_cols].copy()
#     out_path = OUT_DIR / f"steam_reviews_{split_name}_game_prior_features.parquet"
#     split_out.to_parquet(out_path, index=False)
#     print(split_name, len(split_out), out_path)
